# Vivir más sin ser más rico

**Trabajo Práctico Final · Visualización de la Información · ITBA**  
Javier Peral Belmont y Pilar Frutos · 16 de Noviembre de 2026

En 2024, Costa Rica vivía 2,1 años más que Estados Unidos con poco más de un tercio de su PIB por habitante. ¿Qué tan bien describe el ingreso las diferencias de longevidad entre países? Primero miramos el patrón y sus excepciones. Después miramos cómo se relaciona la longevidad con el acceso al agua potable y dónde se concentra la exposición a la contaminación del aire.



In [39]:
!pip -q install babel pycountry

import time

import altair as alt
import numpy as np
import pandas as pd
import pycountry
import requests
from babel import Locale

alt.data_transformers.disable_max_rows()
# Números con coma decimal y punto de miles en ejes y fichas
alt.renderers.set_embed_options(format_locale={"decimal": ",", "thousands": ".", "grouping": [3], "currency": ["$", ""]})

API = "https://data360api.worldbank.org/data360"
DESDE, HASTA = 2000, 2024


In [40]:
def pedir(metodo, ruta, intentos=6, **kwargs):
    """Llama a la API reintentando: a veces responde con el cuerpo vacío o corta la conexión."""
    for intento in range(intentos):
        try:
            r = requests.request(metodo, f"{API}/{ruta}", timeout=120, **kwargs)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            if intento == intentos - 1:
                raise
            time.sleep(2 * (intento + 1))


# 1. Exploración de Data360

Consultamos el catálogo de Data360 del Banco Mundial y contamos los indicadores por base. Elegimos **World Development Indicators (WDI)** porque reúne series anuales por país de economía, población y salud.

In [41]:
respuesta = pedir("POST", "searchv2", json={
    "search": "*",
    "top": 0,
    "count": True,
    "filter": "type eq 'indicator'",
    "facets": ["series_description/database_id,count:1000"],
})

bases = (pd.DataFrame(respuesta["@search.facets"]["series_description/database_id"])
         .rename(columns={"value": "base", "count": "indicadores"})
         .sort_values("indicadores", ascending=False, ignore_index=True))

print("Bases de datos en Data360:", len(bases))
print("Indicadores en total:", bases["indicadores"].sum())
bases.head(15)


Bases de datos en Data360: 171
Indicadores en total: 9850


,base,indicadores
0,WB_WDI,1545
1,IMF_BOP,1146
2,WB_EDSTATS,1072
3,WB_ES,584
4,IMF_IFS,495
5,WB_GS,363
6,WB_FINDEX,280
7,IMF_GFS_SOO,218
8,BS_SGI,193
9,WB_HNP,192


# 2. Selección de indicadores y preparación de los datos

Partimos de cuatro conceptos: longevidad, ingreso por habitante, acceso al agua y exposición a la contaminación del aire. Buscamos cada uno en el catálogo de Data360 con términos en inglés y filtramos por WDI. La tabla muestra las primeras tres coincidencias de cada búsqueda.


In [42]:
busquedas_wdi = [
    "life expectancy at birth",
    "GDP per capita PPP constant",
    "basic drinking water",
    "PM2.5 mean annual exposure",
]
resultados_wdi = []
for consulta in busquedas_wdi:
    respuesta = pedir("POST", "searchv2", json={
        "search": consulta,
        "filter": "type eq 'indicator' and series_description/database_id eq 'WB_WDI'",
        "select": "series_description/idno, series_description/name",
        "top": 3,
    })
    for resultado in respuesta["value"]:
        indicador = resultado["series_description"]
        resultados_wdi.append({
            "Búsqueda": consulta,
            "Código en Data360": indicador["idno"],
            "Nombre en el catálogo": indicador["name"],
        })
coincidencias_wdi = pd.DataFrame(resultados_wdi)
coincidencias_wdi


,Búsqueda,Código en Data360,Nombre en el catálogo
0,life expectancy at birth,WB_WDI_SP_DYN_LE00_IN,"Life expectancy at birth, total (years)"
1,life expectancy at birth,WB_WDI_SP_DYN_LE00_FE_IN,"Life expectancy at birth, female (years)"
2,life expectancy at birth,WB_WDI_SP_DYN_LE00_MA_IN,"Life expectancy at birth, male (years)"
3,GDP per capita PPP constant,WB_WDI_PA_NUS_PPP,"PPP conversion factor, GDP (LCU per internatio..."
4,GDP per capita PPP constant,WB_WDI_NY_GDP_PCAP_PP_KD,"GDP per capita, PPP (constant 2021 internation..."
5,GDP per capita PPP constant,WB_WDI_EG_USE_COMM_GD_PP_KD,"Energy use (kg of oil equivalent) per $1,000 G..."
6,basic drinking water,WB_WDI_SH_H2O_BASW_ZS,People using at least basic drinking water ser...
7,basic drinking water,WB_WDI_SH_H2O_BASW_UR_ZS,People using at least basic drinking water ser...
8,basic drinking water,WB_WDI_SH_H2O_BASW_RU_ZS,People using at least basic drinking water ser...
9,PM2.5 mean annual exposure,WB_WDI_EN_ATM_PM25_MC_M3,"PM2.5 air pollution, mean annual exposure (mic..."


### Por qué elegimos estas series

- **Esperanza de vida al nacer.** Elegimos la serie total para comparar el indicador del país. El catálogo también ofrece series separadas por sexo.
- **PIB por habitante.** Elegimos la serie por persona en paridad de poder adquisitivo (PPA) y a precios constantes de 2021 para comparar niveles de ingreso entre países.
- **Agua potable.** Usamos el porcentaje de toda la población que accede al menos a un servicio básico de agua potable. El catálogo también ofrece series específicas para áreas urbanas y rurales.
- **PM2,5.** Elegimos la exposición media anual, que permite comparar la concentración entre países y años. Verificamos cuántas economías tenían datos de PM2,5 y esperanza de vida en el mismo año. El cruce define el período del mapa.

| Indicador de WDI | Código en Data360 | Unidad |
|---|---|---|
| Esperanza de vida al nacer | `WB_WDI_SP_DYN_LE00_IN` | Años |
| PIB por habitante (PPA) | `WB_WDI_NY_GDP_PCAP_PP_KD` | Dólares internacionales constantes de 2021 |
| Acceso básico a agua potable | `WB_WDI_SH_H2O_BASW_ZS` | Porcentaje de la población |
| Exposición media a partículas finas PM2,5 | `WB_WDI_EN_ATM_PM25_MC_M3` | Microgramos por metro cúbico (µg/m³) |

El acceso básico al agua se refiere al uso de una fuente mejorada a no más de 30 minutos de ida y vuelta. La exposición a PM2,5 es un promedio anual ponderado por población.


### Cómo llega una observación

Este ejemplo muestra el registro de exposición a PM2,5 de Argentina en 2023. La API identifica la economía con `REF_AREA`, el año con `TIME_PERIOD` y el valor con `OBS_VALUE`. Usamos los totales cuando vienen desagregados y cruzamos las series por economía y año.


In [43]:
obs = pedir("GET", "data", params={
    "DATABASE_ID": "WB_WDI",
    "INDICATOR": "WB_WDI_EN_ATM_PM25_MC_M3",
    "REF_AREA": "ARG",
    "TIME_PERIOD": 2023,
})["value"][0]
pd.DataFrame([obs])[["REF_AREA", "TIME_PERIOD", "INDICATOR", "OBS_VALUE", "UNIT_MEASURE"]]


,REF_AREA,TIME_PERIOD,INDICATOR,OBS_VALUE,UNIT_MEASURE
0,ARG,2023,WB_WDI_EN_ATM_PM25_MC_M3,14.346595,MCG_M3


In [44]:
def traer_indicador(codigo, columna):
    """Descarga todas las páginas de un indicador y devuelve una tabla país–año."""
    filas, skip = [], 0
    while True:
        pagina = pedir("GET", "data", params={"DATABASE_ID": "WB_WDI", "INDICATOR": codigo, "top": 1000, "skip": skip})
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"] or not pagina["value"]:
            break
    d = pd.DataFrame(filas)
    for dim in ["SEX", "AGE", "URBANISATION"]:      # nos quedamos con el total si hay desagregaciones
        if dim in d and d[dim].nunique() > 1:
            d = d[d[dim] == "_T"]
    return pd.DataFrame({
        "iso3": d["REF_AREA"],
        "anio": pd.to_numeric(d["TIME_PERIOD"]).astype(int),
        columna: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    }).dropna()


INDICADORES = {
    "vida": "WB_WDI_SP_DYN_LE00_IN",
    "pib": "WB_WDI_NY_GDP_PCAP_PP_KD",
    "agua": "WB_WDI_SH_H2O_BASW_ZS",
    "pm25": "WB_WDI_EN_ATM_PM25_MC_M3",
}

series = {clave: traer_indicador(codigo, clave) for clave, codigo in INDICADORES.items()}


La API incluye agregados como el total mundial. Los excluimos con la lista de economías del Banco Mundial, que también aporta la región y el grupo de ingreso de cada país.

In [45]:
jerarquia = pedir("GET", "portal/v1/hierarchy", params={"type": "country"})["data"]

REGIONES = {
    "East Asia & Pacific": "Asia oriental y Pacífico",
    "Europe & Central Asia": "Europa y Asia central",
    "Latin America & Caribbean": "América Latina y el Caribe",
    "Middle East, North Africa, Afghanistan & Pakistan": "Medio Oriente y norte de África",
    "North America": "América del Norte",
    "South Asia": "Asia meridional",
    "Eastern and Southern Africa": "África subsahariana",
    "Western and Central Africa": "África subsahariana",
}
INGRESOS = {"Low income": "Bajo", "Lower middle income": "Medio bajo", "Upper middle income": "Medio alto", "High income": "Alto"}

regiones = pd.DataFrame([{"iso3": c["id"], "nombre_en": c["name"], "region": REGIONES[g["name"]]}
                         for g in jerarquia["HCL_WB_REGIONS"] for c in g["countries"]])
ingresos = pd.DataFrame([{"iso3": c["id"], "ingreso": INGRESOS.get(g["name"])}
                         for g in jerarquia["HCL_WB_INCOME"] for c in g["countries"]])
paises = regiones.merge(ingresos, on="iso3", how="left")
paises["ingreso"] = paises["ingreso"].fillna("Sin clasificar")

nombres_es = Locale("es").territories
def nombre_es(fila):
    pais = pycountry.countries.get(alpha_3=fila.iso3)
    iso2 = pais.alpha_2 if pais else {"XKX": "XK"}.get(fila.iso3)
    return nombres_es.get(iso2, {"CHI": "Islas del Canal"}.get(fila.iso3, fila.nombre_en))

paises["pais"] = paises.apply(nombre_es, axis=1)

In [46]:
panel = None
for s in series.values():
    panel = s if panel is None else panel.merge(s, on=["iso3", "anio"], how="outer")

panel = (panel[panel["anio"].between(1990, HASTA)]
         .merge(paises[["iso3", "pais", "region", "ingreso"]], on="iso3", how="inner")
         .sort_values(["iso3", "anio"], ignore_index=True))

**Período y cobertura.** El gráfico de PIB usa 2024 y el de agua potable recorre 2000–2024. El mapa llega hasta el último año con datos de PM2,5 y esperanza de vida. La tabla cuenta las economías con ambos datos en años seleccionados.


In [47]:
aire = panel.loc[
    panel["anio"].between(DESDE, HASTA), ["iso3", "pais", "anio", "pm25", "vida"]
].dropna(subset=["pm25", "vida"]).copy()
aire["fecha_anio"] = pd.to_datetime(aire["anio"].astype(str), format="%Y")
ultimo_anio_aire = int(aire["anio"].max())

cobertura_aire = aire.groupby("anio").agg(
    economias=("iso3", "nunique"),
    pm25_mediana=("pm25", "median"),
)
cobertura_aire.rename(columns={
    "economias": "Economías con ambos datos",
    "pm25_mediana": "Mediana de exposición a PM2,5 (µg/m³)",
}).reindex([DESDE, 2010, 2020, ultimo_anio_aire]).round(1)


,Economías con ambos datos,"Mediana de exposición a PM2,5 (µg/m³)"
anio,,
2000,200,22.8
2010,200,20.6
2020,200,17.8
2023,200,17.3


# 3. Costa Rica vive más que Estados Unidos con poco más de un tercio de su ingreso

Cada punto representa una economía en 2024. La línea muestra cuántos años de esperanza de vida estima el modelo para cada nivel de PIB por habitante. Costa Rica queda 4,5 años por encima de lo estimado y Estados Unidos, 3,0 años por debajo.

Pasá el cursor sobre cada país para comparar su esperanza de vida observada con la estimada. Usá *Buscar país* para ver dónde queda otra economía respecto de la línea.

Con los datos de 2024, buscamos la curva que pasa lo más cerca posible del conjunto de puntos. Para calcularla, usamos una escala en la que duplicar el PIB por habitante siempre ocupa la misma distancia.

La curva muestra la esperanza de vida que estima el modelo para cada nivel de PIB por habitante. A medida que aumenta el PIB, la curva se aplana.


In [48]:
corte = panel[panel["anio"] == 2024].dropna(subset=["vida", "pib"]).copy()
b, a = np.polyfit(np.log(corte["pib"]), corte["vida"], 1)
corte["esperada"] = a + b * np.log(corte["pib"])
corte["brecha"] = corte["vida"] - corte["esperada"]
r2 = 1 - corte["brecha"].var() / corte["vida"].var()

In [49]:
%pip -q install country_converter

import altair as alt
import country_converter as coco

alt.data_transformers.disable_max_rows()

columnas = ['iso3', 'anio', 'pais', 'region', 'pib', 'vida']
g1 = panel.loc[panel['pib'].gt(0), columnas].dropna().copy()

# Continentes para el color y regiones del Banco Mundial para el filtro.
nombres_continentes = {
    'Africa': 'África',
    'America': 'América',
    'Asia': 'Asia',
    'Europe': 'Europa',
    'Oceania': 'Oceanía',
    'Antarctica': 'Antártida',
}
catalogo = coco.CountryConverter().data
continente_por_iso3 = (
    catalogo.drop_duplicates('ISO3')
    .set_index('ISO3')['continent']
    .map(nombres_continentes)
    .to_dict()
)
continente_por_iso3.update({'CHI': 'Europa', 'XKX': 'Europa'})

g1['continente'] = g1['iso3'].map(continente_por_iso3).fillna('Sin clasificar')
g1['destacado_inicial'] = g1['pais'].isin([
    'Costa Rica', 'Estados Unidos', 'Nigeria', 'Nauru',
])

paleta = {
    'África': '#8dd3c7',
    'América': '#fdb462',
    'Asia': '#bebada',
    'Europa': '#fb8072',
    'Oceanía': '#80b1d3',
}
continentes_presentes = set(g1['continente'])
continentes = [c for c in paleta if c in continentes_presentes]

# Controles de año, región y búsqueda.
primer_anio = int(g1['anio'].min())
ultimo_anio = int(g1['anio'].max())

anio_pib = alt.param(
    name='anio_pib',
    value=ultimo_anio,
    bind=alt.binding_range(
        min=primer_anio, max=ultimo_anio, step=1, name='Año: ',
    ),
)
region_pib = alt.param(
    name='region_pib',
    value='Todas',
    bind=alt.binding_select(
        options=['Todas'] + sorted(g1['region'].unique().tolist()),
        name='Región: ',
    ),
)
buscar_pib = alt.param(
    name='buscar_pib',
    value='',
    bind=alt.binding(
        input='search', placeholder='Ej.: Costa Rica', name='Buscar país: ',
    ),
)

# Ejes fijos para comparar años y regiones.
pib_max = float(g1['pib'].max()) * 1.03

x = alt.X(
    'pib:Q',
    title='PIB por habitante (miles de dólares internacionales de 2021, PPA)',
    scale=alt.Scale(type='linear', domain=[0, pib_max], nice=False),
    axis=alt.Axis(
        values=list(range(0, int(pib_max) + 1, 25_000)),
        labelExpr="format(datum.value / 1000, '.0f')",
        grid=False,
    ),
)
y = alt.Y(
    'vida:Q',
    title='Esperanza de vida al nacer (años)',
    scale=alt.Scale(domain=[10, 90], nice=False),
    axis=alt.Axis(values=list(range(10, 91, 10))),
)

# Se destaca el nombre completo o una coincidencia parcial única.
consulta = "lower(trim(buscar_pib))"
base = (
    alt.Chart(g1)
    .transform_filter(alt.datum.anio == anio_pib)
    .transform_filter("region_pib == 'Todas' || datum.region == region_pib")
    .transform_calculate(
        exacta=f"lower(datum.pais) == {consulta} ? 1 : 0",
        parcial=(
            f"length({consulta}) >= 2 && "
            f"indexof(lower(datum.pais), {consulta}) >= 0 ? 1 : 0"
        ),
    )
    .transform_joinaggregate(
        total_exactas='sum(exacta)',
        total_parciales='sum(parcial)',
    )
)

coincide_busqueda = (
    "datum.exacta == 1 || "
    "(datum.total_exactas == 0 && datum.total_parciales == 1 && datum.parcial == 1)"
)
sin_coincidencia_unica = "datum.total_exactas == 0 && datum.total_parciales != 1"
mostrar_etiqueta = (
    f"(length({consulta}) == 0 && datum.destacado_inicial) || ({coincide_busqueda})"
)

ficha = [
    alt.Tooltip('pais:N', title='País o territorio'),
    alt.Tooltip('anio:O', title='Año'),
    alt.Tooltip('continente:N', title='Continente'),
    alt.Tooltip('region:N', title='Región del Banco Mundial'),
    alt.Tooltip('vida:Q', title='Esperanza de vida (años)', format='.1f'),
    alt.Tooltip('pib:Q', title='PIB por habitante (PPA de 2021)', format=',.0f'),
]

puntos = base.mark_circle(size=75, stroke='white', strokeWidth=0.7).encode(
    x=x,
    y=y,
    color=alt.Color(
        'continente:N',
        title='Continente',
        scale=alt.Scale(
            domain=continentes,
            range=[paleta[c] for c in continentes],
        ),
        legend=alt.Legend(orient='right', direction='vertical'),
    ),
    opacity=alt.condition(
        f"({sin_coincidencia_unica}) || ({coincide_busqueda})",
        alt.value(0.85),
        alt.value(0.18),
    ),
    tooltip=ficha,
)

etiquetas = (
    base.transform_filter(mostrar_etiqueta)
    .mark_text(align='left', dx=9, dy=-8, fontWeight='bold')
    .encode(x=x, y=y, text='pais:N', tooltip=ficha)
)

grafico_1 = (
    (puntos + etiquetas)
    .properties(
        width='container',
        height=420,
        autosize=alt.AutoSizeParams(type='fit-x', contains='padding'),
        title=alt.Title('PIB por habitante y esperanza de vida', anchor='start'),
    )
    .add_params(anio_pib, region_pib, buscar_pib)
    .configure_view(stroke=None)
)

grafico_1

alt.LayerChart(...)

Nigeria y Nauru quedan por debajo de la línea. Su esperanza de vida es unos 15 y 10 años menor, respectivamente, que la estimada para su nivel de ingreso. ¿Qué vemos cuando comparamos su acceso al agua potable?

# 4. Hasta 24 años de diferencia en longevidad con acceso casi universal al agua

Hasta acá comparamos a los países por su ingreso. Ahora miramos algo más concreto de la vida cotidiana, el acceso al agua potable. Cuando casi toda la población tiene acceso, ¿qué tan parecida es la esperanza de vida de un país a otro?

In [50]:
def datos_anuales(col, decimales):
    d = panel[panel["anio"].between(DESDE, HASTA) & panel[col].notna() & panel["vida"].notna()]
    return d.assign(valor=d[col].round(decimales), vida=d["vida"].round(1))[["iso3", "pais", "region", "ingreso", "anio", "valor", "vida"]]


agua = datos_anuales("agua", 1)
agua["fecha_anio"] = pd.to_datetime(agua["anio"].astype(str), format="%Y")


In [51]:
agua = agua.assign(
    continente=agua["iso3"].map(continente_por_iso3).fillna("Sin clasificar")
)

paleta_agua = {
    "África": "#8dd3c7",
    "América": "#fdb462",
    "Asia": "#bebada",
    "Europa": "#fb8072",
    "Oceanía": "#80b1d3",
}
continentes_agua = list(paleta_agua)
colores_agua = list(paleta_agua.values())

if "Sin clasificar" in agua["continente"].values:
    continentes_agua.append("Sin clasificar")
    colores_agua.append("#666666")

cobertura_agua = agua.groupby("anio", as_index=False)["iso3"].nunique()
cobertura_agua["texto"] = (
    cobertura_agua["anio"].astype(str)
    + " · "
    + cobertura_agua["iso3"].astype(str)
    + " economías con datos de agua y esperanza de vida"
)

anio_agua = alt.param(
    name="anio_agua",
    value=HASTA,
    bind=alt.binding_range(min=DESDE, max=HASTA, step=1, name="Año: "),
)
buscar_agua = alt.param(
    name="buscar_agua",
    value="",
    bind=alt.binding(
        input="search", placeholder="Ej.: Argentina", name="Buscar un país: ",
    ),
)
COINCIDE_AGUA = (
    "length(buscar_agua) > 1 && "
    "indexof(lower(datum.pais), lower(buscar_agua)) >= 0"
)

acceso_min_agua = float(agua["valor"].min())
x_agua = alt.X(
    "valor:Q",
    title="Población con acceso básico al agua potable (%)",
    scale=alt.Scale(type="linear", domain=[acceso_min_agua, 100], nice=False),
    axis=alt.Axis(
        values=[acceso_min_agua] + list(range(30, 101, 10)),
        labelExpr="format(datum.value, '~f') + ' %'",
    ),
)
vida_min_agua = int(agua["vida"].min() // 10) * 10
y_agua = alt.Y(
    "vida:Q",
    title="Esperanza de vida (años)",
    scale=alt.Scale(domain=[vida_min_agua, 88], nice=False),
)

base_agua = alt.Chart(agua).transform_filter(alt.datum.anio == anio_agua)

FICHA_AGUA = [
    alt.Tooltip("pais:N", title="País"),
    alt.Tooltip("continente:N", title="Continente"),
    alt.Tooltip("region:N", title="Región del Banco Mundial"),
    alt.Tooltip("fecha_anio:T", title="Año", format="%Y"),
    alt.Tooltip("valor:Q", title="Acceso básico a agua (% de la población)", format=".1f"),
    alt.Tooltip("vida:Q", title="Esperanza de vida (años)", format=".1f"),
]

color_continente_agua = alt.Color(
    "continente:N",
    title="Continente",
    scale=alt.Scale(domain=continentes_agua, range=colores_agua),
    legend=alt.Legend(
        orient="right", direction="vertical", symbolSize=75, symbolOpacity=1,
    ),
)
opacidad_puntos = (
    alt.when(COINCIDE_AGUA).then(alt.value(1))
    .when("length(buscar_agua) > 1").then(alt.value(0.18))
    .otherwise(alt.value(0.76))
)

puntos_agua = base_agua.mark_circle(
    size=55, stroke="white", strokeWidth=0.45,
).encode(
    x=x_agua,
    y=y_agua,
    color=color_continente_agua,
    opacity=opacidad_puntos,
    tooltip=FICHA_AGUA,
)

# Recorrido del país buscado hasta el año elegido.
recorrido = alt.Chart(agua).transform_filter(
    f"{COINCIDE_AGUA} && datum.anio <= anio_agua"
)
linea_recorrido = recorrido.mark_line(
    color="#262626",
    strokeWidth=1.5,
    point=alt.OverlayMarkDef(size=12, color="#262626"),
).encode(x=x_agua, y=y_agua, detail="pais:N", order="anio:Q")

nombre_recorrido = base_agua.transform_filter(COINCIDE_AGUA).mark_text(
    align="right", dx=-8, dy=-8, fontWeight="bold",
).encode(x=x_agua, y=y_agua, text="pais:N")

encabezado_agua = alt.Chart(cobertura_agua).transform_filter(
    alt.datum.anio == anio_agua
).mark_text(
    align="left", x=0, y=-14, fontSize=12, color="#444444",
).encode(text="texto:N")

casos_agua = base_agua.transform_filter(
    "length(buscar_agua) <= 1 && (datum.iso3 == 'NRU' || datum.iso3 == 'NGA')"
)
aros_agua = casos_agua.mark_point(
    shape="circle", filled=False, size=150, stroke="#262626", strokeWidth=2,
).encode(x=x_agua, y=y_agua, tooltip=FICHA_AGUA)

etiquetas_casos_agua = casos_agua.mark_text(
    align="right", dx=-10, dy=-9, fontWeight="bold", color="#262626",
).encode(x=x_agua, y=y_agua, text="pais:N", tooltip=FICHA_AGUA)

grafico_2 = (
    puntos_agua + linea_recorrido + nombre_recorrido + encabezado_agua
    + aros_agua + etiquetas_casos_agua
).add_params(
    anio_agua, buscar_agua,
).properties(
    width=760,
    height=460,
    padding={"top": 20, "left": 10, "right": 16, "bottom": 10},
)

grafico_2

alt.LayerChart(...)

En 2024, los países donde una mayor proporción de la población tenía acceso básico al agua potable tendían a tener una esperanza de vida más alta.

Nigeria y Nauru siguen ese patrón. En Nigeria, el 82 % de la población tenía acceso y la esperanza de vida era de 55 años. En Nauru, el acceso llegaba al 98 % y la esperanza de vida era de 62 años.

Al comparar a Nauru con otros países donde casi toda la población tenía agua potable, aparece otra diferencia. En Mónaco, con 100 % de acceso, la esperanza de vida llegaba a 86,5 años.

Ahora volvamos a Nigeria. ¿Cómo se compara su esperanza de vida con la de un país con niveles parecidos de contaminación del aire?

# 5. Veinte años de diferencia en esperanza de vida con niveles similares de contaminación del aire

Después de mirar el agua que tomamos, pasamos al aire que respiramos. Para ver cómo se relaciona la contaminación con la longevidad, comparamos la exposición a partículas finas en el aire (PM2,5) con la esperanza de vida de cada país


In [52]:
codigos_mapa = {pais.alpha_3: pais.numeric for pais in pycountry.countries}
datos_mapa_aire = aire.assign(id=aire["iso3"].map(codigos_mapa)).dropna(subset=["id"])
mundo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/world-atlas@2.0.2/countries-110m.json",
    "countries",
)


In [53]:
anio_aire = alt.param(
    name="anio_aire",
    value=ultimo_anio_aire,
    bind=alt.binding_range(min=DESDE, max=ultimo_anio_aire, step=1, name="Año: "),
)
clic_en_pais_aire = "event.item && event.item.datum && isValid(event.item.datum.iso3)"
pais_aire = alt.selection_point(
    name="pais_aire", fields=["iso3"], toggle=False,
    on=f"click[{clic_en_pais_aire}]", clear=f"click[!({clic_en_pais_aire})]",
)
color_aire = alt.Color(
    "pm25:Q", title="Exposición a PM2,5 (µg/m³)",
    scale=alt.Scale(domain=[0, aire["pm25"].max()], scheme="viridis", reverse=True, nice=False),
    legend=alt.Legend(orient="bottom", gradientLength=320, tickCount=6),
)
ficha_aire = [
    alt.Tooltip("pais:N", title="País o territorio"),
    alt.Tooltip("fecha_anio:T", title="Año", format="%Y"),
    alt.Tooltip("pm25:Q", title="Exposición a PM2,5 (µg/m³)", format=".1f"),
    alt.Tooltip("vida:Q", title="Esperanza de vida al nacer (años)", format=".1f"),
]

mapa_base = alt.Chart(mundo).transform_filter(alt.datum.id != "010").mark_geoshape(
    fill="#e6e8eb", stroke="white", strokeWidth=0.45,
)
mapa_aire = alt.Chart(datos_mapa_aire).transform_filter(
    alt.datum.anio == anio_aire
).transform_lookup(
    lookup="id", from_=alt.LookupData(mundo, key="id"), as_="geometria",
).transform_filter(
    "isValid(datum.geometria)"
).mark_geoshape(stroke="white", strokeWidth=0.45, cursor="pointer").encode(
    shape=alt.Shape("geometria:G"),
    color=color_aire,
    opacity=alt.condition(pais_aire, alt.value(1), alt.value(0.22)),
    tooltip=ficha_aire,
).properties(name="paises_mapa_aire").add_params(pais_aire)
mapa = (mapa_base + mapa_aire).project(type="equalEarth").properties(
    width=465, height=290, title=alt.Title("Distribución geográfica", anchor="start"),
)

x_aire = alt.X(
    "pm25:Q", title="Exposición a PM2,5 (µg/m³)",
    scale=alt.Scale(domainMin=0), axis=alt.Axis(grid=False),
)
y_aire = alt.Y(
    "vida:Q", title="Esperanza de vida al nacer (años)",
    scale=alt.Scale(domain=[10, 90], nice=False), axis=alt.Axis(gridColor="#ededed"),
)
base_aire = alt.Chart(aire).transform_filter(alt.datum.anio == anio_aire).encode(
    x=x_aire, y=y_aire,
)
puntos_aire = base_aire.mark_circle(size=48, stroke="white", strokeWidth=0.5, cursor="pointer").encode(
    color=color_aire,
    opacity=alt.condition(pais_aire, alt.value(0.9), alt.value(0.16)),
    tooltip=ficha_aire,
).properties(name="paises_dispersion_aire").add_params(pais_aire)

# Ancla el eje horizontal con el máximo de todo el período.
limite_aire = alt.Chart(aire).transform_aggregate(pm25="max(pm25)").mark_point(opacity=0).encode(x=x_aire)
dispersion_aire = (limite_aire + puntos_aire).properties(
    width=385, height=290, title=alt.Title("Exposición y esperanza de vida", anchor="start"),
)
grafico_3 = alt.hconcat(mapa, dispersion_aire, spacing=22).add_params(
    anio_aire,
).properties(
    padding={"top": 16, "left": 10, "right": 16, "bottom": 10},
).configure_view(stroke=None)
grafico_3


alt.HConcatChart(...)

En 2023, Bangladés y Nigeria aparecen cerca en el eje horizontal. Su exposición a PM2,5 era similar, de 69,3 y 65,7 µg/m³, respectivamente. En el eje vertical, en cambio, los separan 20 años. La esperanza de vida era de 74,7 años en Bangladés y de 54,5 en Nigeria.

Entre las 200 economías con datos ese año, una mayor exposición a PM2,5 tendía a coincidir con una menor esperanza de vida. Esa tendencia era menos marcada que la que vimos con el acceso al agua potable.

# 6. Conclusión

En 2024, Costa Rica tenía poco más de un tercio del PIB por habitante de Estados Unidos y dos años más de esperanza de vida. Esa comparación nos llevó a mirar qué otras condiciones acompañan las diferencias de longevidad entre países.

Al mirar el agua, encontramos que en Nauru casi toda la población tenía acceso básico al agua potable y la esperanza de vida era de 62 años. Otros países con un acceso parecido llegaban a los 86. Al mirar el aire, encontramos otra diferencia llamativa. En 2023, Bangladés y Nigeria tenían niveles similares de exposición a partículas finas, y sus esperanzas de vida estaban separadas por 20 años.

Al comparar Nigeria con Nauru, aparece el patrón que esperaríamos. En Nigeria, menos personas tenían acceso al agua potable y la esperanza de vida también era más baja. Camboya agrega algo a la historia. En 2024, tanto en Nigeria como en Camboya, alrededor del 83 % de la población tenía acceso básico al agua potable. La esperanza de vida era de 55 años en Nigeria y de 71 en Camboya.

Nigeria y Camboya tenían un acceso al agua muy parecido y 16 años de diferencia en esperanza de vida. ¿Qué otras condiciones podrían ayudar a entender esa brecha? Un próximo paso sería comparar qué tan fácil es acceder a la atención médica en cada país.

### Fuentes

- **Datos:** *World Development Indicators*, vía [Data360](https://data360.worldbank.org/en/api). La esperanza de vida proviene de estimaciones de Naciones Unidas y los datos de agua del Programa Conjunto de Monitoreo OMS/UNICEF.
- **Aire:** [Exposición anual a PM2,5](https://data.worldbank.org/indicator/EN.ATM.PM25.MC.M3), Banco Mundial.
- **Geometrías:** [World Atlas](https://github.com/topojson/world-atlas). A esta resolución, algunas economías pequeñas de la tabla de cobertura no tienen representación en el mapa.
